In [1]:
import itertools
import os
from collections import Counter

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from thefuzz import fuzz
from tqdm import tqdm

In [2]:
default_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

In [5]:
registry = pd.read_csv(
    "../20250630-0800-gleif-goldencopy-lei2-golden-copy.csv",
    dtype=str,
    usecols=[
        "LEI",
        "Entity.LegalName",
        "Entity.LegalAddress.FirstAddressLine",
        "Entity.LegalAddress.City",
        "Entity.LegalAddress.Region",
        "Entity.LegalAddress.Country",
        "Entity.LegalAddress.PostalCode",
        "Entity.HeadquartersAddress.FirstAddressLine",
        "Entity.HeadquartersAddress.City",
        "Entity.HeadquartersAddress.Region",
        "Entity.HeadquartersAddress.Country",
        "Entity.HeadquartersAddress.PostalCode",
    ],
)

In [6]:
registry["LegalAddress-region"] = np.where(
    registry["Entity.LegalAddress.Region"].notna(),
    registry["Entity.LegalAddress.Region"],
    registry["Entity.LegalAddress.Country"],
)
registry["HeadquartersAddress-region"] = np.where(
    registry["Entity.HeadquartersAddress.Region"].notna(),
    registry["Entity.HeadquartersAddress.Region"],
    registry["Entity.HeadquartersAddress.Country"],
)

In [7]:
us_or_canada = registry["Entity.LegalAddress.Country"].isin(["US", "CA"]) | registry[
    "Entity.HeadquartersAddress.Country"
].isin(["US", "CA"])

In [14]:
sec2023 = pd.read_csv("../data/sec-2023.csv", dtype={"cik": str})

In [15]:
address_replacement = {
    "STREET": "ST",
    "AVENUE": "AVE",
    "BOULEVARD": "BLVD",
    "LANE": "LN",
    "DRIVE": "DR",
    "ROAD": "RD",
    "CRESCENT": "CRES",
    "PLACE": "PL",
    "TERRACE": "TER",
    "COURT": "CT",
    "CIRCLE": "CIR",
    "SQUARE": "SQ",
    "ALLEY": "ALY",
    "MOUNT": "MT",
    "HILL": "HL",
    "HILLS": "HLS",
    "ESTATE": "EST",
    "ESTATES": "ESTS",
    "GARDEN": "GDN",
    "GARDENS": "GDNS",
    "GREEN": "GRN",
    "GROVE": "GRV",
    "PARKWAY": "PKWY",
    "PARK": "PK",
    "PARKS": "PKS",
    "PARKLAND": "PKLD",
    "MARKET": "MKT",
    "HIGHWAY": "HWY",
    "TOLLWAY": "TLWY",
    "FLAT": "FLT",
    "SUITE": "STE",
    "TOWER": "TWR",
    "BUILDING": "BLDG",
    "BLOCK": "BLK",
    "APARTMENT": "APT",
    "FLOOR": "FLR",
    "INDUSTRIAL": "IND",
    "CENTER": "CTR",
    "COMPLEX": "CMPLX",
    "UNIVERSITY": "UNIV",
    "INSTITUTE": "INST",
    "PLAZA": "PLZ",
    "TRAIL": "TRL",
    "BRIDGE": "BRG",
    "EAST": "E",
    "WEST": "W",
    "SOUTH": "S",
    "NORTH": "N",
    "POINT": "PT",
    "PENTHOUSE": "PH",
    "SAINT": "ST",
    "SAINTS": "STS",
    "JUNCTION": "JCT",
    "CROSSING": "XING",
    "EXPRESSWAY": "EXPY",
    "FREEWAY": "FWY",
    "EXTENSION": "EXT",
    "MEADOWS": "MDWS",
    "FIELDS": "FLDS",
    "FIELD": "FLD",
    "WOODS": "WDS",
    "FOREST": "FRST",
    "ROOM": "RM",
    "FIRST": "1ST",
    "SECOND": "2ND",
    "THIRD": "3RD",
    "FOURTH": "4TH",
    "FIFTH": "5TH",
    "SIXTH": "6TH",
    "SEVENTH": "7TH",
    "EIGHTH": "8TH",
    "NINTH": "9TH",
    "TENTH": "10TH",
    "ELEVENTH": "11TH",
    "TWELVTH": "12TH",
    "ONE": "1",
    "TWO": "2",
    "THREE": "3",
    "FOUR": "4",
    "FIVE": "5",
    "SIX": "6",
    "SEVEN": "7",
    "EIGHT": "8",
    "NINE": "9",
    "TEN": "10",
    "ELEVEN": "11",
    "TWELVE": "12",
    "THIRTEEN": "13",
    "FOURTEEN": "14",
    "FIFTEEN": "15",
    "SIXTEEN": "16",
    "SEVENTEEN": "17",
    "EIGHTEEN": "18",
    "NINTEEN": "19",
    "TWENTY": "20",
}


def normalize_address(address):
    if not isinstance(address, str):
        return ""
    return " ".join(
        address_replacement.get(word, word)
        for word in address.upper()
        .replace("POST OFFICE", "PO")
        .replace("P.O", "PO")
        .replace(".", " ")
        .replace(",", " ")
        .replace("-", " ")
        .split()
        if word != "C/O"
    )

In [16]:
us_codes = {
    "ALABAMA": "AL",
    "KENTUCKY": "KY",
    "OHIO": "OH",
    "ALASKA": "AK",
    "LOUISIANA": "LA",
    "OKLAHOMA": "OK",
    "ARIZONA": "AZ",
    "MAINE": "ME",
    "OREGON": "OR",
    "ARKANSAS": "AR",
    "MARYLAND": "MD",
    "PENNSYLVANIA": "PA",
    "AMERICAN SAMOA": "AS",
    "MASSACHUSETTS": "MA",
    "PUERTO RICO": "PR",
    "CALIFORNIA": "CA",
    "MICHIGAN": "MI",
    "RHODE ISLAND": "RI",
    "COLORADO": "CO",
    "MINNESOTA": "MN",
    "SOUTH CAROLINA": "SC",
    "CONNECTICUT": "CT",
    "MISSISSIPPI": "MS",
    "SOUTH DAKOTA": "SD",
    "DELAWARE": "DE",
    "MISSOURI": "MO",
    "TENNESSEE": "TN",
    "DISTRICT OF COLUMBIA": "DC",
    "MONTANA": "MT",
    "TEXAS": "TX",
    "FLORIDA": "FL",
    "NEBRASKA": "NE",
    "TRUST TERRITORIES": "TT",
    "GEORGIA": "GA",
    "NEVADA": "NV",
    "UTAH": "UT",
    "GUAM": "GU",
    "NEW HAMPSHIRE": "NH",
    "VERMONT": "VT",
    "HAWAII": "HI",
    "NEW JERSEY": "NJ",
    "VIRGINIA": "VA",
    "IDAHO": "ID",
    "NEW MEXICO": "NM",
    "VIRGIN ISLANDS": "VI",
    "ILLINOIS": "IL",
    "NEW YORK": "NY",
    "WASHINGTON": "WA",
    "INDIANA": "IN",
    "NORTH CAROLINA": "NC",
    "WEST VIRGINIA": "WV",
    "IOWA": "IA",
    "NORTH DAKOTA": "ND",
    "WISCONSIN": "WI",
    "KANSAS": "KS",
    "NORTHERN MARIANA ISLANDS": "MP",
    "WYOMING": "WY",
}
ca_codes = {
    "ALBERTA": "AB",
    "BRITISH COLUMBIA": "BC",
    "MANITOBA": "MB",
    "NEW BRUNSWICK": "NB",
    "NEWFOUNDLAND AND LABRADOR": "NL",
    "NORTHWEST TERRITORIES": "NT",
    "NOVA SCOTIA": "NS",
    "NUNAVUT": "NU",
    "ONTARIO": "ON",
    "PRINCE EDWARD ISLAND": "PE",
    "QUEBEC": "QC",
    "QUÉBEC": "QC",
    "SASKATCHEWAN": "SK",
    "YUKON": "YT",
}


def states_to_codes(x):
    if not isinstance(x, str):
        return ""
    y = " ".join(x.upper().replace(".", " ").replace(",", " ").split())
    if y in us_codes.values():
        return f"US-{y}"
    if y in ca_codes.values():
        return f"CA-{y}"
    z = us_codes.get(y)
    if z is not None:
        return f"US-{z}"
    z = ca_codes.get(y)
    if z is not None:
        return f"CA-{z}"
    if y == "CAYMAN ISLANDS":
        return "KY"
    assert False

In [11]:
columns = {
    "LEI": "LEI",
    "Entity.LegalName": "name",
    "Entity.LegalAddress.FirstAddressLine": "legal_address",
    "Entity.LegalAddress.City": "legal_city",
    "LegalAddress-region": "legal_region",
    "Entity.LegalAddress.PostalCode": "legal_zip",
    "Entity.HeadquartersAddress.FirstAddressLine": "hq_address",
    "Entity.HeadquartersAddress.City": "hq_city",
    "HeadquartersAddress-region": "hq_region",
    "Entity.HeadquartersAddress.PostalCode": "hq_zip",
}
gleif = registry[us_or_canada][columns.keys()].rename(columns=columns)
gleif["name"] = gleif["name"].str.upper()
gleif["legal_address"] = gleif["legal_address"].apply(normalize_address)
gleif["legal_city"] = gleif["legal_city"].str.upper()
gleif["legal_region"] = gleif["legal_region"].str.upper()
gleif["legal_zip"] = gleif["legal_zip"].str.upper().fillna("")
gleif["hq_address"] = gleif["hq_address"].apply(normalize_address)
gleif["hq_city"] = gleif["hq_city"].str.upper()
gleif["hq_region"] = gleif["hq_region"].str.upper()
gleif["hq_zip"] = gleif["hq_zip"].str.upper().fillna("")
gleif

,LEI,name,legal_address,legal_city,legal_region,legal_zip,hq_address,hq_city,hq_region,hq_zip
0,001GPB6A9XPE8XJICC14,FIDELITY ADVISOR LEVERAGED COMPANY STOCK FUND,245 SUMMER ST,BOSTON,US-MA,02210,FIDELITY MANAGEMENT & RESEARCH COMPANY LLC,WILMINGTON,US-MA,19801
1,004L5FPTUREIWK9T2N63,"HUTCHIN HILL CAPITAL, LP",CORPORATION SERVICE COMPANY,WILMINGTON,US-DE,19808,22ND FLR,NEW YORK,US-NY,10106
2,00EHHQ2ZHDCFXJCPCL46,VANGUARD FIDUCIARY TRUST COMPANY VANGUARD RUSS...,VANGUARD FIDUCIARY TRUST COMPANY VANGUARD FINA...,MALVERN,US-PA,19355,VANGUARD FIDUCIARY TRUST COMPANY VANGUARD FIDU...,MALVERN,US-PA,19355
3,00GBW0Z2GYIER7DHDS71,"ARISTEIA CAPITAL, L.L.C.",THE CORPORATION TRUST COMPANY,WILMINGTON,US-DE,19801,3RD FLR,GREENWICH,US-CT,06830
4,00KLB2PFTM3060S2N216,OAKMARK INTERNATIONAL FUND,C T CORPORATION SYSTEM,BOSTON,US-MA,02110,HARRIS ASSOCIATES L P,CHICAGO,US-IL,60606
...,...,...,...,...,...,...,...,...,...,...
2994233,ZZG891NILC4TOKRZ0S19,FRANKLIN BISSETT TREASURY BILL FUND,FRANKLIN TEMPLETON INVESTMENTS CORP,TORONTO,CA-ON,M5H 3T4,200 KING ST W,TORONTO,CA-ON,M5H 3T4
2994234,ZZKNSWIGOMECFE0H8C92,GRAHAM K4D TRADING LTD.,OGIER GLOBAL (BVI) LIMITED,ROAD TOWN,VG,VG1110,40 HIGHLAND AVE,NORWALK,US-CT,06853
2994235,ZZM83464VMIMD4PJGL80,GMO FIXED INCOME ABSOLUTE RETURN FUND (ONSHORE...,THE CORPORATION TRUST COMPANY,WILMINGTON,US-DE,19801,GRANTHAM MAYO VAN OTTERLOO & CO LLC,BOSTON,US-MA,02109
2994237,ZZTCRPWYDHXBFN8Q2W40,1001 K STREET ASSOCIATES LIMITED PARTNERSHIP,C RICHARD BEYDA,WASHINGTON,US-DC,20036,C RICHARD BEYDA,WASHINGTON,US-DC,20036


In [18]:
sec = pd.DataFrame(
    {
        "CIK": sec2023["cik"].fillna(""),
        "sec_name": sec2023["name"].str.upper(),
        "sec_address": sec2023["address"].apply(normalize_address),
        "sec_city": sec2023["city"].apply(
            lambda x: (
                " ".join(x.upper().replace(".", " ").replace(",", " ").split())
                if isinstance(x, str)
                else ""
            )
        ),
        "sec_state": sec2023["state"].apply(states_to_codes),
        "sec_zip": sec2023["zip"].str.upper().fillna(""),
    }
)
sec

,CIK,sec_name,sec_address,sec_city,sec_state,sec_zip
0,0001750149,"INHIBIKASE THERAPEUTICS, INC.",3350 RIVERWOOD PKWY SE,ATLANTA,US-GA,30339
1,0001375195,THE CORETEC GROUP INC.,600 S WAGNER RD,ANN ARBOR,US-MI,48103
2,0001726445,"SEER, INC.",3800 BRG PKWY,REDWOOD CITY,US-CA,94065
3,0001678660,PRELUDE THERAPEUTICS INC,200 POWDER MILL RD,WILMINGTON,US-DE,19803
4,0001705843,"CALYXT, INC.",2800 MT RIDGE RD,ROSEVILLE,US-MN,55113-1127
...,...,...,...,...,...,...
7976,0001709401,"RUBIUS THERAPEUTICS, INC.",VERDOLINO & LOWEY P C,FOXBOROUGH,US-MA,02035
7977,0001023128,LITHIA MOTORS INC,150 N BARTLETT ST,MEDFORD,US-OR,97501
7978,0000891014,MINERALS TECHNOLOGIES INC.,622 3RD AVE,NEW YORK,US-NY,10017-6707
7979,0001030469,OFG BANCORP,254 MUÑOZ RIVERA AVE,SAN JUAN,,00918


In [13]:
del registry
del sec2023

In [21]:
match_legal = gleif.merge(sec, left_on=["legal_address", "legal_city", "legal_region"], right_on=["sec_address", "sec_city", "sec_state"])

In [23]:
match_legal[["name", "sec_name"]]

,name,sec_name
0,FIDELITY ADVISOR LEVERAGED COMPANY STOCK FUND,FIDELITY PRIVATE CREDIT FUND
1,LAZARD FRERES & CO QUANTITATIVE EQUITY FUND,LAZARD GROUP LLC
2,TRUSTMARK NATIONAL BANK,TRUSTMARK CORP
3,HOMESTREET BANK,"HOMESTREET, INC."
4,PIMCO REALESTATEPLUS COLLECTIVE TRUST,SEI INVESTMENTS COMPANY
...,...,...
17137,US TREASURY INFLATION PROTECTED SECURITIES FUND E,ISHARES GOLD TRUST MICRO
17138,FIDELITY MASSACHUSETTS MUNICIPAL TRUST,FIDELITY PRIVATE CREDIT FUND
17139,FIDELITY COMMONWEALTH TRUST - FIDELITY SMALL C...,FIDELITY PRIVATE CREDIT FUND
17140,PRIVATE EQUITY PARTNERS 2004 MGR LP,GITLAB INC.


In [24]:
match_hq = gleif.merge(sec, left_on=["hq_address", "hq_city", "hq_region"], right_on=["sec_address", "sec_city", "sec_state"])

In [26]:
match_hq[["name", "sec_name"]]

,name,sec_name
0,LAZARD FRERES & CO QUANTITATIVE EQUITY FUND,LAZARD GROUP LLC
1,TRUSTMARK NATIONAL BANK,TRUSTMARK CORP
2,HOMESTREET BANK,"HOMESTREET, INC."
3,LAURION CAPITAL GLOBAL MARKETS MASTER FUND LTD.,PAVMED INC.
4,LAURION CAPITAL GLOBAL MARKETS MASTER FUND LTD.,LUCID DIAGNOSTICS INC.
...,...,...
96949,"JANE STREET GLOBAL TRADING, LLC",BROOKFIELD REAL ESTATE INCOME TRUST INC.
96950,"JANE STREET GLOBAL TRADING, LLC",BROOKFIELD DTLA FUND OFFICE TRUST INVESTOR INC.
96951,"AMAZON.COM, INC.","AMAZON.COM, INC."
96952,MERRILL LYNCH BANK AND TRUST COMPANY (CAYMAN) ...,PIMCO CAPITAL SOLUTIONS BDC CORP.


In [32]:
all_matches = pd.concat([
    match_legal[["LEI", "CIK"]].assign(matched_legal=1, matched_hq=0),
    match_hq[["LEI", "CIK"]].assign(matched_legal=0, matched_hq=1),
]).groupby(["LEI", "CIK"]).sum().reset_index()

In [33]:
all_matches

,LEI,CIK,matched_legal,matched_hq
0,001GPB6A9XPE8XJICC14,0001920453,1,0
1,01E7JU5W846RIZXBMX63,0001326141,1,1
2,01J4SO3XTWZF4PP38209,0000036146,1,1
3,01KWVG908KE7RKPTNP46,0001518715,1,1
4,03L7Q0KIHFQXHO0DKZ47,0000350894,1,0
...,...,...,...,...
105217,ZXOVMTW1OS6Z6XPSLX60,0001713407,0,1
105218,ZXTILKJKG63JELOEG630,0001018724,0,1
105219,ZXTNNR37O4UOIGYDYY55,0001905824,0,1
105220,ZYGTNJW8XLETUHK28M72,0001883788,0,1


In [34]:
all_matches.to_csv("lei-cik-matches.csv")

In [35]:
all_matches[["matched_legal", "matched_hq"]].sum()

matched_legal    17142
matched_hq       96954
dtype: int64